# GraphRAG (with LlamaIndex)

- GraphRAG은 RAG와 QFS(Query Focused Summarization, 쿼리 중심 요약)의 장점을 결합한 기술.
  - large text dataset에 대한 복잡한 쿼리를 효과적으로 처리함.
- RAG는 정확한 정보를 가져오는데 성능이 좋지만, 주제에 대한 이해가 필요한 (비교적 범위가 넓은) 쿼리에 대해서는 성능이 떨어짐.
  - 그리고 QFS는 이런 문제를 해결해주긴 하지만, scalable 하지 않음.
- GraphRAG은 이 두가지 방법을 결합한 기술: [project link](https://www.microsoft.com/en-us/research/project/graphrag/)

_example repo에서도 언급하듯이, 이 notebook은 GraphRAG의 approximate implementation._


## GraphRAG Approach

GraphRAG은 2 step으로 이뤄짐.

1. Graph Generation: 주어진 문서를 기반으로 그래프를 생성하고, community와 community의 summary를 만듬.
2. Answer to the Query: 1단계에서 생성된 community의 summary를 활용해서 주어진 query에 대답.

**Graph Generation**

1. Source documents to Text chunks
   - 원분 문서를 더 쉽게 처리할 수 있도록 작은 text chunk로 나눔
2. Text chunks to Element instances
   - 각 text chunk를 분석해서 entity와 relationship을 식별하고 추출
   - 이를 활용해 해당 요소들을 나타내는 tuple list를 생성
3. Element instances to Element summaries
   - 추출된 entity와 relationship은 LLM을 사용해 각 요소에 대한 descriptive text block으로 요약됨
4. Element summaries to Graph communities
   - entity, relationship에 대한 요약은 graph를 형성
   - Hierarchical Leiden 알고리즘을 사용해 계층 구조를 만들고 community로 분할
5. Graph communities to Community summaries
   - LLM은 각 community에 대한 요약본을 생성, dataset의 전반적인 주제 구조와 semantic에 대한 insight를 제공

**Answering the Query**

Community summaries to Global answers

- community 요약은 user query에 응답하는데 활용.
- 중간 답변을 생성한 후, 이를 통합하여 global answer를 도출하는 단계가 포함되어 있음


## GraphRAG pipeline components

앞서 언급한 모든 과정을 구축하기 위해, 아래와 같은 구성 요소들을 구현

- **Source documents to text chunks**
  - (위의 1 과정)
  - `SentenceSplitter`를 사용, chunk size를 1024로, chunk overlap을 20으로 설정
- **Text chunks to Element instances, Element instances to Element summaries**
  - (위의 2~3 과정)
  - `GraphRAGExtrator`를 이용
- **Element summaries to Graph communities and Graph communities to Community summaries**
  - (위의 4~5 과정)
  - `GraphRAGStore`를 이용
- **Community summaries to Global answers**
  - (위의 5 ~ 마지막 과정)
  - `GraphQueryEngine` 이용


Community building을 위해 hierarchical_leiden 알고리즘을 사용해야 하는데, 이는 `graspologic` 패키지에서 제공함.

_numpy 1.26.4에 최적화 된 패키지인 듯. 기존에 설치된 2.1.3이 downgrade 되었음..._


## Load Data

- Github에 공개된 Diffbot에서 가져온 뉴스 기사 샘플 데이터를 활용.
- 총 2500개의 샘플
  - 편의를 위해 기사의 제목과 본문을 포함한 50개의 샘플을 사용함.


In [1]:
import pandas as pd
from llama_index.core import Document

news = pd.read_csv(
    "https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/news_articles.csv"
)[:50]

news.head()

,title,date,text
0,Chevron: Best Of Breed,2031-04-06T01:36:32.000000000+00:00,JHVEPhoto Like many companies in the O&G secto...
1,FirstEnergy (NYSE:FE) Posts Earnings Results,2030-04-29T06:55:28.000000000+00:00,FirstEnergy (NYSE:FE – Get Rating) posted its ...
2,Dáil almost suspended after Sinn Féin TD put p...,2023-06-15T14:32:11.000000000+00:00,The Dáil was almost suspended on Thursday afte...
3,Epic’s latest tool can animate hyperrealistic ...,2023-06-15T14:00:00.000000000+00:00,"Today, Epic is releasing a new tool designed t..."
4,"EU to Ban Huawei, ZTE from Internal Commission...",2023-06-15T13:50:00.000000000+00:00,The European Commission is planning to ban equ...


In [2]:
# LlamaIndex에 맞춰서 document 준비
documents = [
    Document(text=f"{row['title']}: {row['text']}")
    for i, row in news.iterrows()
]

## Setup API Key & LLM


In [3]:
import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model='gemini-2.5-flash', api_key=GEMINI_API_KEY)

## GraphRAGExtractor

- triplet(subject-relation-object)를 추출하고, LLM을 사용해 엔티티와 관계에 대한 설명을 해당 속성에 추가.
  - triplet의 정보가 좀 더 풍부해지는 셈.
- 기능 자체는 `SimpleLLMPathExtractor`와 유사하지만, 엔티티 및 관계의 설명을 처리하기 위한 추가적인 구현이 있음.

주요 기능을 살펴보면...

**Key compoments**

1. `llm`: extract에 활용할 LLM
2. `extract_prompt`: LLM이 정보를 추출하는데 도움을 주기 위한 prompt template
3. `parse_fn`: LLM의 출력을 구조화된 데이터로 파싱하는 함수
4. `max_paths_per_chunk`: text chunk당 추출되는 triplet의 개수를 제한
5. `num_workers`: 여러개의 text node를 병렬로 처리하기 위함

**Main methods**

1. `__call__`: text node의 리스트를 처리하기 위한 entry point
2. `acall`: call의 비동기 버전
3. `_aextract`: 각 node를 처리하는 핵심 method

**Extraction process**

입력된 각 node 마다 (즉, 각 text chunk 마다)

1. text와 extraction prompt를 함께 LLM에 전달
2. LLM의 응답을 분석해서 엔티티, 관계, 엔티티 및 관계에 대한 설명을 추출
3. 엔티티는 EntityNode 객체로 변환되고, 엔티티 설명은 metadata에 저장
4. 관계는 Relation object로 변환되고, 관계 설명은 metadata에 저장
5. 이 값들이 이제 node metadata에 `KG_NODES_KEY`,`KG_RELATIONS_KEY`로 추가됨

_어떻게 보면 Knowledge Graph를 구축하는 과정._


In [4]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

from typing import Any, List, Callable, Optional, Union, Dict
from IPython.display import Markdown, display

from llama_index.core.async_utils import run_jobs
from llama_index.core.indices.property_graph.utils import (
    default_parse_triplets_fn,
)
from llama_index.core.graph_stores.types import (
    EntityNode,
    KG_NODES_KEY,
    KG_RELATIONS_KEY,
    Relation,
)
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.prompts.default_prompts import (
    DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
)
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.bridge.pydantic import BaseModel, Field


class GraphRAGExtractor(TransformComponent):
    """
    graph로부터 triplet(subject, relation, object)을 추출
    """

    llm: LLM
    extract_prompt: PromptTemplate
    parse_fn: Callable
    num_workers: int
    max_paths_per_chunk: int

    def __init__(self, llm: Optional[LLM]=None, extract_prompt: Optional[Union[str, PromptTemplate]] = None,
                 parse_fn: Callable = default_parse_triplets_fn, max_paths_per_chunk: int = 10,
                 num_workers: int = 4) -> None:
        """Init param"""
        from llama_index.core import Settings

        if isinstance(extract_prompt, str):
            extract_prompt = PromptTemplate(extract_prompt)

        super().__init__(
            llm=llm or Settings.llm,
            extract_prompt=extract_prompt or DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
            parse_fn=parse_fn,
            num_workers=num_workers,
            max_paths_per_chunk=max_paths_per_chunk,
        )

    @classmethod
    def class_name(cls) -> str:
        return "GraphExtractor"
    
    def __call__(self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any) -> List[BaseNode]:
        """node로부터 triplet 추출"""
        return asyncio.run(
            self.acall(nodes, show_progress=show_progress, **kwargs)
        )
    
    async def _aextract(self, node: BaseNode) -> BaseNode:
        """node로부터 triplet 추출(비동기)"""
        assert hasattr(node, "text")

        text = node.get_content(metadata_mode="llm")
        try:
            # prompt와 text를 기반으로 응답 받고
            llm_response = await self.llm.apredict(
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )

            # entity, relationship에 넣기
            entities, entities_relationship = self.parse_fn(llm_response)
        except ValueError:
            entities = []
            entities_relationship = []

        # 이미 존재하는 값이면 pop
        existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        metadata = node.metadata.copy()

        # 추출한 엔티티(triplet)를 loop 하면서 metadata 생성
        for entity, entity_type, description in entities:
            metadata[
                "entity_description"
            ] = description

            entity_node = EntityNode(
                name=entity, label=entity_type, properties=metadata
            )

            existing_nodes.append(entity_node)

        metadata = node.metadata.copy()
        for triple in entities_relationship:
            subj, rel, obj, description = triple
            subj_node = EntityNode(name=subj, properties=metadata)
            obj_node = EntityNode(name=obj, properties=metadata)
            metadata["relationship_description"] = description
            rel_node = Relation(
                label=rel,
                source_id=subj_node.id,
                target_id=obj_node.id,
                properties=metadata,
            )

            existing_nodes.extend([subj_node, obj_node])
            existing_relations.append(rel_node)

        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node

    async def acall(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """node로부터 triplet을 추출"""
        jobs = []
        for node in nodes:
            jobs.append(self._aextract(node))

        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting paths from text",
        )


## GraphRAGStore

- GraphRAG 파이프라인을 구현하도록 설계된 LlamaIndex의 `SimplePropertyGraphStore`를 확장한 버전.
- community detection 알고리즘을 사용
  - 그래프에서 관련된 노드를 그룹화 한 다음,
  - LLM을 사용해 각 커뮤니티에 대한 요약을 생성

**Key Methods**

- `build_communities()`
  1. 그래프를 `NetworkX` 그래프로 변환
  2. community detection 알고리즘 사용(Hierarchical Leiden algorithm)
  3. 각 커뮤니티의 세부 정보 수집
  4. 각 커뮤니티의 요약 생성

- `generate_community_summary(text)`
  - LLM을 활용해 커뮤니티 안에 있는 관계의 요약을 생성
  - 이 요약에는 엔티티 이름과 관계 설명의 내용이 포함되어 있음.

- `_create_nx_graph()`
  - `NetworkX` 그래프로 변환하는 함수

- `_collect_community_info(nx_graph, clusters)`
  - 각 node의 커뮤니티를 기반으로, node에 대한 detail information을 수집
  - 커뮤니티 안에서의 관계를 string으로 표현

- `_summarize_communities(community_info)`
  - LLM을 사용해 각 커뮤니티에 대한 요약을 생성

- `get_community_summaries()`
  - 커뮤니티 요약 정보가 생성되지 않은 경우, 생성해서 return


In [5]:
import re
from llama_index.core.graph_stores import SimplePropertyGraphStore
import networkx as nx
from graspologic.partition import hierarchical_leiden

from llama_index.core.llms import ChatMessage


class GraphRAGStore(SimplePropertyGraphStore):
    community_summary = {}
    max_cluster_size = 5

    def generate_community_summary(self, text):
        """Generate summary for a given text using an LLM."""
        messages = [
            ChatMessage(
                role="system",
                content=(
                    "You are provided with a set of relationships from a knowledge graph, each represented as "
                    "entity1->entity2->relation->relationship_description. Your task is to create a summary of these "
                    "relationships. The summary should include the names of the entities involved and a concise synthesis "
                    "of the relationship descriptions. The goal is to capture the most critical and relevant details that "
                    "highlight the nature and significance of each relationship. Ensure that the summary is coherent and "
                    "integrates the information in a way that emphasizes the key aspects of the relationships."
                ),
            ),
            ChatMessage(role="user", content=text),
        ]
        response = GoogleGenAI().chat(messages)
        clean_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return clean_response

    def build_communities(self):
        """Builds communities from the graph and summarizes them."""
        nx_graph = self._create_nx_graph()
        community_hierarchical_clusters = hierarchical_leiden(
            nx_graph, max_cluster_size=self.max_cluster_size
        )
        community_info = self._collect_community_info(
            nx_graph, community_hierarchical_clusters
        )
        self._summarize_communities(community_info)

    def _create_nx_graph(self):
        """Converts internal graph representation to NetworkX graph."""
        nx_graph = nx.Graph()
        for node in self.graph.nodes.values():
            nx_graph.add_node(str(node))
        for relation in self.graph.relations.values():
            nx_graph.add_edge(
                relation.source_id,
                relation.target_id,
                relationship=relation.label,
                description=relation.properties["relationship_description"],
            )
        return nx_graph

    def _collect_community_info(self, nx_graph, clusters):
        """Collect detailed information for each node based on their community."""
        community_mapping = {item.node: item.cluster for item in clusters}
        community_info = {}
        for item in clusters:
            cluster_id = item.cluster
            node = item.node
            if cluster_id not in community_info:
                community_info[cluster_id] = []

            for neighbor in nx_graph.neighbors(node):
                if community_mapping[neighbor] == cluster_id:
                    edge_data = nx_graph.get_edge_data(node, neighbor)
                    if edge_data:
                        detail = f"{node} -> {neighbor} -> {edge_data['relationship']} -> {edge_data['description']}"
                        community_info[cluster_id].append(detail)
        return community_info

    def _summarize_communities(self, community_info):
        """Generate and store summaries for each community."""
        for community_id, details in community_info.items():
            details_text = (
                "\n".join(details) + "."
            )  # Ensure it ends with a period
            self.community_summary[
                community_id
            ] = self.generate_community_summary(details_text)

    def get_community_summaries(self):
        """Returns the community summaries, building them if not already done."""
        if not self.community_summary:
            self.build_communities()
        return self.community_summary

c:\Users\skdbs\.conda\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## GraphRAGQueryEngine

- GraphRAG에서 제안된 방식을 사용해 쿼리를 처리하는 엔진
- `GraphRAGStore`에서 생성된 커뮤니티 요약을 활용해서 사용자 쿼리에 답변.

**Main Components**

- `graph_store`: 커뮤니티 요약을 담고 있는 `GraphRAGStore` 인스턴스.
- `llm`: 답변을 생성하고 aggregate 하는데 사용되는 LLM.

**Key Methods**

- `custom_query(query_str: str)`
  - 쿼리 처리를 하는 main entry point.
  - 커뮤니티 요약을 검색하고, 각 요약에서 답변을 생성한 후, 답변들을 종합해서 최종 답변을 생성

- `generate_answer_from_summary(community_summary, query)`
  - 하나의 커뮤니티 요약을 기반으로, 쿼리에 대한 답변을 생성.
  - LLM을 사용해 query context에 맞게 커뮤니티 요약을 해석

- `aggregate_answers(community_answers)`
  - 서로 다른 커뮤니티의 개별 답변을 종합, 일관성 있는 최종 답변을 제시
  - LLM을 사용해 다양한 관점을 하나의 간결한 답변으로 종합.

**Query Processing Flow**

1. graph store에서 커뮤니티 요약을 retrieve
2. 각 커뮤니티 요약마다, query에 대한 구체적인 답변을 생성
3. 각 커뮤니티의 답변을 종합, 일관성있는 최종 답변을 생성


In [ ]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.llms import LLM


class GraphRAGQueryEngine(CustomQueryEngine):
    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query_str: str) -> str:
        """Process all community summaries to generate answers to a specific query."""
        community_summaries = self.graph_store.get_community_summaries()
        community_answers = [
            self.generate_answer_from_summary(community_summary, query_str)
            for _, community_summary in community_summaries.items()
        ]

        final_answer = self.aggregate_answers(community_answers)
        return final_answer

    def generate_answer_from_summary(self, community_summary, query):
        """Generate an answer from a community summary based on a given query using LLM."""
        prompt = (
            f"Given the community summary: {community_summary}, "
            f"how would you answer the following query? Query: {query}"
        )
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content="I need an answer based on the above information.",
            ),
        ]
        response = self.llm.chat(messages)
        cleaned_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return cleaned_response

    def aggregate_answers(self, community_answers):
        """Aggregate individual community answers into a final, coherent response."""
        # intermediate_text = " ".join(community_answers)
        prompt = "Combine the following intermediate answers into a final, concise response."
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content=f"Intermediate answers: {community_answers}",
            ),
        ]
        final_response = self.llm.chat(messages)
        cleaned_final_response = re.sub(
            r"^assistant:\s*", "", str(final_response)
        ).strip()
        return cleaned_final_response

## Build end-to-end GraphRAG pipeline

필요 구성 요소는 전부 만들었으니 GraphRAG pipeline을 만들면 된다.

1. text로 부터 node/chunk 생성
2. `GraphRAGExtractor`와 `GraphRAGStore`를 사용해서 `PropertyGraphIndex`를 구성
3. 앞서 구축한 그래프를 이용해서 커뮤니티를 만들고, 각 커뮤니티에 대한 요약을 생성
4. `GraphRAGQueryEngine`을 써서 querying 시작


### text로부터 node/chunk 생성


In [6]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=20,
)

nodes = splitter.get_nodes_from_documents(documents)

print(len(nodes))

50


### `GraphRAGExtractor`와 `GraphRAGStore`를 사용해서 `PropertyGraphIndex`를 구성


In [7]:
KG_TRIPLET_EXTRACT_TMPL = """
-Goal-
Given a text document, identify all entities and their entity types from the text and all relationships among the identified entities.
Given the text, extract up to {max_knowledge_triplets} entity-relation triplets.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: Type of the entity
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"$$$$<entity_name>$$$$<entity_type>$$$$<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relation: relationship between source_entity and target_entity
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other

Format each relationship as ("relationship"$$$$<source_entity>$$$$<target_entity>$$$$<relation>$$$$<relationship_description>)

3. When finished, output.

-Real Data-
######################
text: {text}
######################
output:"""


entity_pattern = r'\("entity"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
relationship_pattern = r'\("relationship"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'


def parse_fn(response_str: str) -> Any:
    entities = re.findall(entity_pattern, response_str)
    relationships = re.findall(relationship_pattern, response_str)
    return entities, relationships


kg_extractor = GraphRAGExtractor(
    llm=llm,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=2,
    parse_fn=parse_fn,
)

In [8]:
from llama_index.core import PropertyGraphIndex

index = PropertyGraphIndex(
    nodes=nodes,
    property_graph_store=GraphRAGStore(),
    kg_extractors=[kg_extractor],
    show_progress=True,
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.997189051s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

새 project를 만들어서 거기서 key를 발급받아야 할 듯; 무료라 그런지 제한이 너무 걸린다
